# Qwen-Image 2.1 Uncensored (GGUF) — Colab runner

**Before you start:** Runtime → Change runtime type → **GPU** (T4 works; L4 / A100 are faster).

Run the 3 cells in order:
1. **Setup** – installs ComfyUI + GGUF node, downloads the model files (~15 GB, once per session)
2. **Start engine** – launches ComfyUI in the background
3. **Generate** – type prompt, pick options, press ▶. Images are saved to `/content/outputs`

After the first two cells are done, you only ever re-run cell 3.

In [ ]:
#@title 1️⃣ Setup: install ComfyUI + download model (run once, ~10 min)
#@markdown Choose the files to download. Defaults = recommended low-memory setup.
gguf_quant = "Q4_K_M" #@param ["Q4_0","Q4_K_M","Q5_K_M","Q6_K","Q8_0"]
text_encoder = "int8_convrot (9.3 GB, recommended)" #@param ["int8_convrot (9.3 GB, recommended)","bf16 (17.5 GB, needs a lot of RAM)"]

import os, sys, json, subprocess

def sh(cmd, cwd=None):
    print("$", cmd)
    r = subprocess.run(cmd, shell=True, cwd=cwd)
    if r.returncode != 0:
        raise RuntimeError("Command failed: " + cmd)

# --- GPU check
g = subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
                   shell=True, capture_output=True, text=True)
if g.returncode != 0 or not g.stdout.strip():
    raise RuntimeError("No GPU found. Go to Runtime > Change runtime type > select a GPU, then re-run.")
print("GPU:", g.stdout.strip())

# --- ComfyUI (latest, Qwen-Image 2.1 needs a recent build)
if not os.path.exists("/content/ComfyUI"):
    sh("git clone --depth 1 https://github.com/comfyanonymous/ComfyUI /content/ComfyUI")
else:
    sh("git pull", cwd="/content/ComfyUI")
sh(sys.executable + " -m pip install -q -r requirements.txt", cwd="/content/ComfyUI")

# --- ComfyUI-GGUF: must be the leejet fork (has Qwen-Image 2.1 support)
cn = "/content/ComfyUI/custom_nodes/ComfyUI-GGUF"
if not os.path.exists(cn):
    sh("git clone --depth 1 https://github.com/leejet/ComfyUI-GGUF " + cn)
else:
    sh("git pull", cwd=cn)
sh(sys.executable + " -m pip install -q -r requirements.txt", cwd=cn)

# --- Download model files
sh(sys.executable + ' -m pip install -q "huggingface_hub>=1.5.0,<2.0" hf_xet')
from huggingface_hub import hf_hub_download

REPO = "abenzerps/Qwen-Image-2.1-Uncensored-GGUF"
M = "/content/ComfyUI/models"
gguf_file = "qwen-image-2.1-UC-" + gguf_quant + ".gguf"
te_file = ("qwen3vl_8b_int8_convrot.safetensors" if text_encoder.startswith("int8")
           else "qwen3vl_8b_bf16.safetensors")
vae_file = "qwen_image_2.1_vae_bf16.safetensors"

jobs = [
    (gguf_file,                    M + "/diffusion_models"),
    ("text_encoders/" + te_file,   M),
    ("vae/" + vae_file,            M),
]
for fn, dest in jobs:
    os.makedirs(dest, exist_ok=True)
    print("Downloading", fn, "...")
    hf_hub_download(repo_id=REPO, filename=fn, local_dir=dest)

json.dump({"gguf": gguf_file, "te": te_file, "vae": vae_file},
          open("/content/qwen_cfg.json", "w"))
print("\n✅ Setup done. Now run cell 2.")


In [ ]:
#@title 2️⃣ Start the ComfyUI engine (run after cell 1, ~1 min)
low_vram_mode = False #@param {type:"boolean"}
#@markdown Turn on `low_vram_mode` only if you get "out of memory" errors.
restart_engine = False #@param {type:"boolean"}

import os, sys, time, subprocess, requests

URL = "http://127.0.0.1:8188"

def alive():
    try:
        return requests.get(URL + "/system_stats", timeout=2).ok
    except Exception:
        return False

if restart_engine:
    subprocess.run("pkill -f 'main.py --listen'", shell=True)
    time.sleep(3)

if alive():
    print("Engine already running ✅")
else:
    os.makedirs("/content/outputs", exist_ok=True)
    cmd = [sys.executable, "main.py", "--listen", "127.0.0.1", "--port", "8188",
           "--output-directory", "/content/outputs", "--disable-auto-launch"]
    if low_vram_mode:
        cmd.append("--lowvram")
    log = open("/content/comfyui.log", "w")
    subprocess.Popen(cmd, cwd="/content/ComfyUI", stdout=log, stderr=subprocess.STDOUT)
    print("Starting engine", end="")
    for _ in range(150):
        if alive():
            break
        print(".", end="", flush=True)
        time.sleep(2)
    else:
        print("\n❌ Engine did not start. Last log lines:\n")
        print("".join(open("/content/comfyui.log").readlines()[-40:]))
        raise RuntimeError("ComfyUI failed to start")
    print("\nEngine started ✅")

# --- verify required nodes exist
for node in ["UnetLoaderGGUF", "TextEncodeQwenImage21"]:
    ok = node in requests.get(URL + "/object_info/" + node).json()
    print(("✅ " if ok else "❌ ") + node)
    if not ok:
        print("   Missing! Log tail:")
        print("".join(open("/content/comfyui.log").readlines()[-30:]))
print("\nNow run cell 3.")


In [ ]:
#@title 3️⃣ Generate image
prompt = "make image of icecream" #@param {type:"string"}
style = "Natural" #@param ["None","Natural","Cinematic","Photorealistic","3D Render","Animated / Cartoon","Anime","Digital Painting","Oil Painting","Watercolor","Pixel Art","Cyberpunk","Fantasy Art","Product Photo","Minimal Vector"]
aspect_ratio = "16:9" #@param ["1:1","16:9","9:16","4:3","3:4","3:2","2:3","21:9"]
megapixels = "1.0" #@param ["0.5","1.0","1.5","2.0"]
#@markdown 1.0 MP ≈ 1024×1024. On a free T4, stay at 1.0–1.5.
num_images = 1 #@param {type:"slider", min:1, max:8, step:1}
seed = -1 #@param {type:"integer"}
#@markdown seed = -1 means random.
#@markdown ---
#@markdown Advanced (defaults are the model's recommended values):
steps = 25 #@param {type:"slider", min:4, max:60, step:1}
cfg = 1.0 #@param {type:"number"}
negative_prompt = "" #@param {type:"string"}
#@markdown Note: negative prompt has no effect while cfg = 1.0. Higher cfg tends to burn the image.
sampler_name = "euler" #@param ["euler","euler_ancestral","dpmpp_2m","dpmpp_2m_sde","heun","uni_pc"]
scheduler = "simple" #@param ["simple","normal","karras","beta","sgm_uniform"]

import os, json, math, random, time, requests
from PIL import Image
from IPython.display import display

URL = "http://127.0.0.1:8188"
OUT_DIR = "/content/outputs"

STYLES = {
    "None": "",
    "Natural": "natural photograph, soft natural daylight, true-to-life colors, realistic textures, candid look, no heavy filters",
    "Cinematic": "cinematic film still, dramatic lighting, shallow depth of field, anamorphic lens, subtle film grain, color graded",
    "Photorealistic": "ultra realistic photograph, natural lighting, sharp focus, highly detailed, 85mm lens",
    "3D Render": "high quality 3D render, octane render, soft global illumination, subsurface scattering, detailed materials",
    "Animated / Cartoon": "stylized 3D animated movie look, expressive, vibrant colors, clean shapes, soft lighting",
    "Anime": "anime style illustration, clean line art, vibrant colors, detailed background, cel shading",
    "Digital Painting": "digital painting, concept art, painterly brush strokes, rich colors, highly detailed",
    "Oil Painting": "classical oil painting, visible brush strokes, rich texture, museum quality",
    "Watercolor": "watercolor illustration, soft washes, paper texture, delicate colors",
    "Pixel Art": "pixel art, 16-bit retro game style, limited color palette, crisp pixels",
    "Cyberpunk": "cyberpunk aesthetic, neon lights, rainy night, futuristic city, high contrast",
    "Fantasy Art": "epic fantasy art, magical atmosphere, intricate detail, volumetric light",
    "Product Photo": "professional product photography, studio lighting, clean background, sharp focus",
    "Minimal Vector": "flat vector illustration, minimal design, bold shapes, clean colors",
}

def size_from(ratio, mp):
    a, b = [int(x) for x in ratio.split(":")]
    target = float(mp) * 1_000_000
    h = math.sqrt(target * b / a)
    w = h * a / b
    snap = lambda v: max(256, int(round(v / 32)) * 32)   # multiples of 32
    return snap(w), snap(h)

def build_text_node(full_prompt, neg):
    """Use the model's own text-encode node; fill its inputs from the live schema."""
    info = requests.get(URL + "/object_info/TextEncodeQwenImage21").json().get("TextEncodeQwenImage21")
    if not info:
        # fallback: plain CLIPTextEncode
        nodes = {
            "4": {"class_type": "CLIPTextEncode", "inputs": {"text": full_prompt, "clip": ["2", 0]}},
            "5": {"class_type": "CLIPTextEncode", "inputs": {"text": neg, "clip": ["2", 0]}},
        }
        return nodes, ["4", 0], ["5", 0]

    inputs, prompt_set = {}, False
    for name, spec in info["input"].get("required", {}).items():
        t = spec[0]
        opts = spec[1] if len(spec) > 1 and isinstance(spec[1], dict) else {}
        if t == "CLIP":
            inputs[name] = ["2", 0]
        elif t == "VAE":
            inputs[name] = ["3", 0]
        elif t == "STRING":
            if "neg" in name.lower():
                inputs[name] = neg
            elif not prompt_set:
                inputs[name] = full_prompt
                prompt_set = True
            else:
                inputs[name] = opts.get("default", "")
        elif isinstance(t, list):
            inputs[name] = opts.get("default", t[0])
        elif t == "COMBO":
            options = opts.get("options", [])
            inputs[name] = opts.get("default", options[0] if options else None)
        elif t in ("INT", "FLOAT", "BOOLEAN"):
            inputs[name] = opts.get("default", False if t == "BOOLEAN" else 0)
        else:
            print("⚠️ Unhandled required input on text node:", name, t)
    outs = info.get("output", [])
    cond = [i for i, t in enumerate(outs) if t == "CONDITIONING"]
    pos_i = cond[0] if cond else 0
    neg_i = cond[1] if len(cond) > 1 else pos_i
    return {"4": {"class_type": "TextEncodeQwenImage21", "inputs": inputs}}, ["4", pos_i], ["4", neg_i]

def wait_for(pid, timeout=1500):
    t0 = time.time()
    while time.time() - t0 < timeout:
        h = requests.get(URL + "/history/" + pid).json()
        if pid in h:
            st = h[pid].get("status", {})
            if st.get("status_str") == "error":
                for kind, data in st.get("messages", []):
                    if kind == "execution_error":
                        print("❌", data.get("exception_type"), "-", data.get("exception_message"))
                raise RuntimeError("Generation failed (see message above). Try low_vram_mode in cell 2 or a smaller size.")
            if h[pid].get("outputs"):
                return h[pid]
        time.sleep(2)
    raise TimeoutError("Timed out waiting for the image.")

# ---------- run ----------
if not prompt.strip():
    raise ValueError("Please type a prompt.")
try:
    requests.get(URL + "/system_stats", timeout=3)
except Exception:
    raise RuntimeError("Engine is not running. Run cell 2 first.")

cfgfile = json.load(open("/content/qwen_cfg.json"))
extra = STYLES[style]
full_prompt = prompt.strip() + (", " + extra if extra else "")
W, H = size_from(aspect_ratio, megapixels)
print("Prompt :", full_prompt)
print("Size   : %dx%d | steps %d | cfg %s" % (W, H, steps, cfg))
print("(First image loads the model into memory — can take 2-5 min. Later ones are much faster.)\n")

text_nodes, pos_ref, neg_ref = build_text_node(full_prompt, negative_prompt)
graph = {
    "1": {"class_type": "UnetLoaderGGUF", "inputs": {"unet_name": cfgfile["gguf"]}},
    "2": {"class_type": "CLIPLoader", "inputs": {"clip_name": cfgfile["te"], "type": "qwen_image"}},
    "3": {"class_type": "VAELoader", "inputs": {"vae_name": cfgfile["vae"]}},
    "6": {"class_type": "EmptyLatentImage", "inputs": {"width": W, "height": H, "batch_size": 1}},
    "7": {"class_type": "KSampler", "inputs": {
        "model": ["1", 0], "positive": pos_ref, "negative": neg_ref, "latent_image": ["6", 0],
        "seed": 0, "steps": int(steps), "cfg": float(cfg),
        "sampler_name": sampler_name, "scheduler": scheduler, "denoise": 1.0}},
    "8": {"class_type": "VAEDecode", "inputs": {"samples": ["7", 0], "vae": ["3", 0]}},
    "9": {"class_type": "SaveImage", "inputs": {"images": ["8", 0], "filename_prefix": "qwen_image"}},
}
graph.update(text_nodes)

saved = []
for i in range(int(num_images)):
    s = (int(seed) + i) if int(seed) >= 0 else random.randint(0, 2**32 - 1)
    graph["7"]["inputs"]["seed"] = s
    t0 = time.time()
    r = requests.post(URL + "/prompt", json={"prompt": graph})
    if r.status_code != 200:
        print("❌ ComfyUI rejected the workflow:\n", r.text)
        raise RuntimeError("Workflow rejected — copy the message above and send it to get it fixed.")
    res = wait_for(r.json()["prompt_id"])
    for node_out in res["outputs"].values():
        for img in node_out.get("images", []):
            path = os.path.join(OUT_DIR, img.get("subfolder", ""), img["filename"])
            saved.append(path)
            print("✅ [%d/%d] seed %d | %.0fs | saved: %s" % (i + 1, num_images, s, time.time() - t0, path))
            im = Image.open(path)
            preview = im.copy()
            preview.thumbnail((768, 768))
            display(preview)

print("\nAll images are in", OUT_DIR, "(left sidebar → 📁 Files → outputs)")
